# General Split

In [1]:
import pandas as pd
import numpy as np
import os
import re
import pickle
import json
import torch
from tqdm import tqdm

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from collections import defaultdict

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import itertools
from sklearn.model_selection import KFold

## Step1: data preparation

In [2]:
valid_pattern = re.compile(r'^[ACDEFGHIKLMNPQRSTVWYX]+$')
def is_valid_sequence(seq):
    return bool(valid_pattern.match(seq))
base_data_dir = '/path/to/dataset/'

def load_all_data():
    """
    Step1: data preparation
    """
    Ab_Ag_Aff_data = pd.read_csv(os.path.join(base_data_dir, 'Ab_Ag_Pairs.csv'))

    
    """
    Step2: prepare deltaLogKD matrix, 60620 x 60620
    row - col = logKD_row - logKD_col
    """
    deltaLogKD_filepath = os.path.join(base_data_dir, "matrix_deltaLogKD.npy")
    if not os.path.exists(deltaLogKD_filepath):
        log_Aff_new_values = Ab_Ag_Aff_data['log_Aff_new'].values
        deltaLogKD_matrix = log_Aff_new_values[:, np.newaxis] - log_Aff_new_values[np.newaxis, :]
        deltaLogKD_matrix = deltaLogKD_matrix.astype(np.float32)
        np.save(deltaLogKD_filepath, deltaLogKD_matrix)
        print(f"Finished saving matrix_deltaLogKD.npy")
    else:
        log_Aff_new_values = Ab_Ag_Aff_data['log_Aff_new'].values
        deltaLogKD_matrix = np.load(deltaLogKD_filepath)
        print(f"Load matrix_deltaLogKD.npy")



    """
    Step3: prepare Ag_similarity/distance matrix, 60620 x 60620
    """
    # considering the computation time, which may take around seven days, we have a separate script to compute the distance matrix for Ag_seq, 
    # here we just read the pre-computed distance matrix
    allAgdis_filepath = os.path.join(base_data_dir, "Ag_seq_distance.npy")
    allAgseq_filepath = os.path.join(base_data_dir, "Ag_seq_distance_sequences.pkl")
    allAg_seq_distance = np.load(allAgdis_filepath)
    with open(allAgseq_filepath, 'rb') as f:
        allAg_seq_set = pickle.load(f)
    allAg_seq_distance_matrix = pd.DataFrame(allAg_seq_distance, index=allAg_seq_set, columns=allAg_seq_set)


    Agdis_filepath = os.path.join(base_data_dir, "matrix_Agseq_distance.npy")
    if not os.path.exists(Agdis_filepath):
        Ag_seq_list = Ab_Ag_Aff_data['Ag_seq'].tolist()
        Ag_seq_distance_matrix = allAg_seq_distance_matrix.loc[Ag_seq_list, Ag_seq_list].values
        Ag_seq_distance_matrix = Ag_seq_distance_matrix.astype(np.float32)
        np.save(Agdis_filepath, Ag_seq_distance_matrix)
        print(f"Finished saving matrix_Agseq_distance.npy")
    else:
        Ag_seq_distance_matrix = np.load(Agdis_filepath)
        print(f"Load matrix_Agseq_distance.npy")


    """
    Step4: prepare Ab_similarity matrix, 60620 x 60620, each item refers to a binary value indicating whether the two Ab sequences are identical. 
    If identical, the value is 1, otherwise it is 0.
    """
    Abiden_filepath = os.path.join(base_data_dir, "matrix_Abseq_thesame.npy")
    if not os.path.exists(Abiden_filepath):
        Ab_seq_list = Ab_Ag_Aff_data['Ab_seq'].tolist()
        Ab_seq_array = np.array(Ab_seq_list)
        Ab_seq_identity_matrix = (Ab_seq_array[:, None] == Ab_seq_array[None, :])
        Ab_seq_identity_matrix = Ab_seq_identity_matrix.astype(np.uint8)
        np.save(Abiden_filepath, Ab_seq_identity_matrix)
        print(f"Finished saving matrix_Abseq_thesame.npy")
    else:
        Ab_seq_identity_matrix = np.load(Abiden_filepath)
        print(f"Load matrix_Abseq_thesame.npy")



    """
    Step5: prepare dbID_matrix, |Ag_seq|(4265) x |Ab_seq|(22691)
    """
    dbID_filepath = os.path.join(base_data_dir, "matrix_dbID.npy")
    if not os.path.exists(dbID_filepath):
        Ag_seq_set = sorted(list(set(Ab_Ag_Aff_data["Ag_seq"])))
        Ab_seq_set = sorted(list(set(Ab_Ag_Aff_data["Ab_seq"])))
        Ab_Ag_Aff_data["Ag_seq_idx"] = pd.Categorical(Ab_Ag_Aff_data["Ag_seq"], categories=Ag_seq_set).codes
        Ab_Ag_Aff_data["Ab_seq_idx"] = pd.Categorical(Ab_Ag_Aff_data["Ab_seq"], categories=Ab_seq_set).codes
        N_Ag = len(Ag_seq_set)
        N_Ab = len(Ab_seq_set)

        dbID_matrix = np.full((N_Ag, N_Ab), -1, dtype=np.int32)
        ag_indices = Ab_Ag_Aff_data["Ag_seq_idx"].to_numpy()
        ab_indices = Ab_Ag_Aff_data["Ab_seq_idx"].to_numpy()
        db_values  = Ab_Ag_Aff_data["dbID"].to_numpy(dtype=np.int32)
        dbID_matrix[ag_indices, ab_indices] = db_values
        np.save(dbID_filepath, dbID_matrix)
        print(f"Finished saving matrix_dbID.npy")            
    else:
        Ag_seq_set = sorted(list(set(Ab_Ag_Aff_data["Ag_seq"])))
        Ab_seq_set = sorted(list(set(Ab_Ag_Aff_data["Ab_seq"])))
        Ab_Ag_Aff_data["Ag_seq_idx"] = pd.Categorical(Ab_Ag_Aff_data["Ag_seq"], categories=Ag_seq_set).codes
        Ab_Ag_Aff_data["Ab_seq_idx"] = pd.Categorical(Ab_Ag_Aff_data["Ab_seq"], categories=Ab_seq_set).codes
        N_Ag = len(Ag_seq_set)
        N_Ab = len(Ab_seq_set)

        dbID_matrix = np.load(dbID_filepath)
        print(f"Load matrix_dbID.npy")

    return Ag_seq_set, Ab_seq_set, dbID_matrix, Ab_Ag_Aff_data, deltaLogKD_matrix, Ag_seq_distance_matrix, Ab_seq_identity_matrix
Ag_seq_set, Ab_seq_set, dbID_matrix, Ab_Ag_Aff_data, deltaLogKD_matrix, Ag_seq_distance_matrix, Ab_seq_identity_matrix = load_all_data()


Load matrix_deltaLogKD.npy
Load matrix_Agseq_distance.npy
Load matrix_Abseq_thesame.npy
Load matrix_dbID.npy


## Step2: functions for sampling

In [4]:
def data_sampling(data_set, data_deltaLogKD, data_Ag_seq_distance, data_Ab_seq_identity, data_mode,
                    KD_threshold=0.5, max_dbid_occurrence=10, repeat_sample=2, seed=42):
    """
    Construct Listwise Ab-Ag pairs

    for the Ab-Ag complex seed pair, sampling
    1) Ab_x-Ag-pos
    2) Ab_y-Ag-neg
    3) Ab_x-Ag_var-pos
    4) Ab_y-Ag_var-neg
    """
    N = data_set.shape[0]
    dbIDs = data_set['dbID'].values
    log_Aff_new_values = data_set['log_Aff_new'].values
    dbid_counter = defaultdict(int)
    pairs = [] 

    for re in range(repeat_sample):
        print(f"Repeat sample {re+1}/{repeat_sample}")
        rng = np.random.default_rng(seed + re)
        start_i = 1

        for i in tqdm(range(start_i, N), total=N, desc="Constructing Ab-Ag pairs"):
            obj_row = i

            dkd_row = data_deltaLogKD[i, :i]
            ag_row = data_Ag_seq_distance[i, :i]
            ab_row = data_Ab_seq_identity[i, :i]
            available_js = [j for j in range(i) if dbid_counter[dbIDs[j]] < max_dbid_occurrence]

            if re%2 == 0: 
                # rule1: Abx-Ag-pos, Aby_Ag-neg, |·|>KD_threshold
                mask1 = (np.abs(dkd_row) > KD_threshold) & (ag_row == 0) & (ab_row == 0) & (dkd_row > 0)
                mask1 &= np.isin(range(i), available_js)
                rule1_pos_js = np.where(mask1)[0]
                rule1_pos_sample = rng.choice(rule1_pos_js, size=1, replace=False) if len(rule1_pos_js) > 0 else []
                if len(rule1_pos_sample)==0:
                    mask1 = (np.abs(dkd_row) > KD_threshold) & (ag_row > 0) & (dkd_row > 0)
                    rule1_pos_js = np.where(mask1)[0]
                    rule1_pos_sample = [rule1_pos_js[np.argmin(ag_row[rule1_pos_js])]] if len(rule1_pos_js) > 0 else []

                mask2 = (np.abs(dkd_row) > KD_threshold) & (ag_row == 0) & (ab_row == 0) & (dkd_row < 0)
                mask2 &= np.isin(range(i), available_js)
                rule1_neg_js = np.where(mask2)[0]
                rule1_neg_sample = rng.choice(rule1_neg_js, size=1, replace=False) if len(rule1_neg_js) > 0 else []
                if len(rule1_neg_sample)==0: 
                    mask2 = (np.abs(dkd_row) > KD_threshold) & (ag_row > 0) & (dkd_row < 0)
                    rule1_neg_js = np.where(mask2)[0]
                    rule1_neg_sample = [rule1_neg_js[np.argmin(ag_row[rule1_neg_js])]] if len(rule1_neg_js) > 0 else []



                # rule2: Abx-Agvar-pos, Aby-Agvar-neg, |·|>KD_threshold
                mask3 = (np.abs(dkd_row) > KD_threshold) & (ag_row > 0) & (dkd_row > 0)
                mask3 &= np.isin(range(i), available_js)
                rule2_pos_js = np.where(mask3)[0]
                if len(rule2_pos_js) > 0:
                    ag_values = ag_row[rule2_pos_js]
                    min_ag = ag_values.min()
                    threshold = np.percentile(ag_values, 20)
                    candidates = rule2_pos_js[(ag_values <= threshold) & (ag_values != min_ag)]
                    rule2_pos_sample = rng.choice(candidates, size=1, replace=False) if len(candidates) > 0 else []
                else:
                    rule2_pos_sample = []

                mask4 = (np.abs(dkd_row) > KD_threshold) & (ag_row > 0) & (dkd_row < 0)
                mask4 &= np.isin(range(i), available_js)
                rule2_neg_js = np.where(mask4)[0]
                if len(rule2_neg_js) > 0:
                    ag_values = ag_row[rule2_neg_js]
                    min_ag = ag_values.min()
                    threshold = np.percentile(ag_values, 20)
                    candidates = rule2_neg_js[(ag_values <= threshold) & (ag_values != min_ag)]
                    rule2_neg_sample = rng.choice(candidates, size=1, replace=False) if len(candidates) > 0 else []
                else:
                    rule2_neg_sample = []

                samples = list(rule1_pos_sample) + list(rule1_neg_sample) + list(rule2_pos_sample) + list(rule2_neg_sample) + [obj_row]

            elif re%2 == 1:
                threshold = np.percentile(ag_row, 80)
                mask = (np.abs(dkd_row) > KD_threshold) & (ag_row >= threshold)
                mask &= np.isin(range(i), available_js)

                pos_js = np.where(mask & (dkd_row > 0))[0]
                neg_js = np.where(mask & (dkd_row < 0))[0]

                pos_sample = rng.choice(pos_js, size=2, replace=False) if len(pos_js) >= 2 else []
                neg_sample = rng.choice(neg_js, size=2, replace=False) if len(neg_js) >= 2 else []

                samples = list(pos_sample) + list(neg_sample) + [obj_row]


            if len(samples) == 5 and len(set(samples)) == 5 and all(dbid_counter[dbIDs[s]] < max_dbid_occurrence for s in samples):
                for s in samples:
                    dbid_counter[dbIDs[s]] += 1

                pairs.append({
                    "dbID1": dbIDs[samples[0]],
                    "dbID2": dbIDs[samples[1]],
                    "dbID3": dbIDs[samples[2]],
                    "dbID4": dbIDs[samples[3]],
                    "dbID5": dbIDs[samples[4]],
                    "LogKD1": log_Aff_new_values[samples[0]],
                    "LogKD2": log_Aff_new_values[samples[1]],
                    "LogKD3": log_Aff_new_values[samples[2]],
                    "LogKD4": log_Aff_new_values[samples[3]],
                    "LogKD5": log_Aff_new_values[samples[4]],
                })

    base_df = pd.DataFrame(pairs, columns=["dbID1", "dbID2", "dbID3", "dbID4", "dbID5", "LogKD1", "LogKD2", "LogKD3", "LogKD4", "LogKD5"])
    print(f"Base collection pairs for {data_mode}: ", base_df.shape)


    final_df = base_df
    print(f"Final collection pairs for {data_mode}: ", final_df.shape)
    return list(
        zip(
            base_df["dbID1"],
            base_df["dbID2"],
            base_df["dbID3"],
            base_df["dbID4"],
            base_df["dbID5"],
            base_df["LogKD1"],
            base_df["LogKD2"],
            base_df["LogKD3"],
            base_df["LogKD4"],
            base_df["LogKD5"]
        )
    )

## Step3: Split data

In [5]:
save_dir = os.path.join(base_data_dir, "runs_CV")
fold_idx = 0
n_folds = 5
seed = 42

val_fold = fold_idx
test_fold = (fold_idx + 1) % n_folds
train_folds = [i for i in range(n_folds) if i not in [val_fold, test_fold]]

all_data = Ab_Ag_Aff_data
filtered_indices = all_data.index.values

if not os.path.exists(os.path.join(save_dir, f"fold_0_indices.npy")):
    print(f"\033[31mSpliting data...\033[0m")
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    all_folds = list(kf.split(filtered_indices))
    for i, (_, indices) in enumerate(all_folds):
        np.save(os.path.join(save_dir, f"fold_{i}_indices.npy"), filtered_indices[indices])

    train_indices, val_indices, test_indices = [], [], []
    for i, (_, idx) in enumerate(all_folds):
        if i == val_fold:
            val_indices = filtered_indices[idx]
        elif i == test_fold:
            test_indices = filtered_indices[idx]
        elif i in train_folds:
            train_indices.extend(filtered_indices[idx])

elif os.path.exists(os.path.join(save_dir, f"fold_0_indices.npy")):
        print(f"\033[31mLoading the splited data...\033[0m")
        all_folds = []
        for i in range(n_folds):
            indices = np.load(os.path.join(save_dir, f"fold_{i}_indices.npy"))
            all_folds.append((None, indices))

        train_indices, val_indices, test_indices = [], [], []
        for i, (_, idx) in enumerate(all_folds):
            if i == val_fold:
                val_indices = idx
            elif i == test_fold:
                test_indices = idx
            elif i in train_folds:
                train_indices.extend(idx)
print(f"Data sampling for general split, len of train {len(train_indices)}, validation {len(val_indices)}, test {len(test_indices)}")

Loading the splited data...
Data sampling for general split, len of train 36372, validation 12124, test 12124


In [7]:
print(f"\033[31mSampling data...\033[0m")
train_set = Ab_Ag_Aff_data.iloc[train_indices].reset_index(drop=True)
train_deltaLogKD = deltaLogKD_matrix[np.ix_(train_indices, train_indices)]
train_Ag_seq_distance = Ag_seq_distance_matrix[np.ix_(train_indices, train_indices)]
train_Ab_seq_identity = Ab_seq_identity_matrix[np.ix_(train_indices, train_indices)]


val_set = Ab_Ag_Aff_data.iloc[val_indices].reset_index(drop=True)
val_deltaLogKD = deltaLogKD_matrix[np.ix_(val_indices, val_indices)]
val_Ag_seq_distance = Ag_seq_distance_matrix[np.ix_(val_indices, val_indices)]
val_Ab_seq_identity = Ab_seq_identity_matrix[np.ix_(val_indices, val_indices)]



test_set = Ab_Ag_Aff_data.iloc[test_indices].reset_index(drop=True)
test_deltaLogKD = deltaLogKD_matrix[np.ix_(test_indices, test_indices)]
test_Ag_seq_distance = Ag_seq_distance_matrix[np.ix_(test_indices, test_indices)]
test_Ab_seq_identity = Ab_seq_identity_matrix[np.ix_(test_indices, test_indices)]


train_sampling = data_sampling(train_set, train_deltaLogKD, train_Ag_seq_distance, train_Ab_seq_identity, "train", repeat_sample=4, seed=42, KD_threshold=0.5)
val_sampling = data_sampling(val_set, val_deltaLogKD, val_Ag_seq_distance, val_Ab_seq_identity, "val", repeat_sample=2, seed=42, KD_threshold=0.5)
test_sampling = data_sampling(test_set, test_deltaLogKD, test_Ag_seq_distance, test_Ab_seq_identity, "test", repeat_sample=2, seed=42, KD_threshold=0.5)


data_sampling_dict = {"trainset": train_sampling,  "valset": val_sampling, "testset": test_sampling}
data_sampling_path = os.path.join(save_dir, f"fold_{fold_idx}_data_sampling_split.json")
with open(data_sampling_path, "w") as f:
    json.dump(data_sampling_dict, f, indent=4)

Sampling data...
Repeat sample 1/4


Constructing Ab-Ag pairs: 100%|█████████▉| 36371/36372 [05:29<00:00, 110.39it/s]


Repeat sample 2/4


Constructing Ab-Ag pairs: 100%|█████████▉| 36371/36372 [02:19<00:00, 261.07it/s]


Repeat sample 3/4


...

Base collection pairs for val:  (13095, 10)
Final collection pairs for val:  (13095, 10)
Repeat sample 1/2


Constructing Ab-Ag pairs: 100%|█████████▉| 12123/12124 [00:41<00:00, 294.00it/s]


Repeat sample 2/2


Constructing Ab-Ag pairs: 100%|█████████▉| 12123/12124 [00:17<00:00, 684.36it/s]


Base collection pairs for test:  (13030, 10)
Final collection pairs for test:  (13030, 10)
